In [ ]:
# GPU + version checks
import torch, ultralytics

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")
print("Ultralytics:", ultralytics.__version__)

In [ ]:
# model training

from ultralytics import YOLO

# Fine-tune more using the last weights
model = YOLO("D:/floorplan-app/floorplanner-cost-estimate/training_model/runs/detect/train/weights/last.pt")

# model = YOLO("yolov8m.pt")

model.train(
    data="../dataset.yaml",
    epochs=20,
    imgsz=1024,
    batch=8,
    workers=4,
    project="../runs/detect",
    name="train",
    exist_ok=True
)
# for resuming training
# model = YOLO("D:/floorplan-app/floorplanner-cost-estimate/training_model/runs/detect/train/weights/last.pt")
# model.train(resume=True)

In [ ]:
# test trained model (make sure to put the best.pt file in "models/trained")
from ultralytics import YOLO
import cv2
import os

model = YOLO("D:/floorplan-app/floorplanner-cost-estimate/training_model/models/best.pt")
 
# Save folder for YOLO output
save_dir = "runs/detect/predict"
os.makedirs(save_dir, exist_ok=True)
 
results = model.predict(
    source=["../data/images/predict/25.10.13  MECH (1)_page-0009.jpg","../data/images/predict/25.10.31  MECH_page-0004.jpg"],
    stream=True,   # custom draw mode
)
 
index = 0
for r in results:
    img = r.orig_img.copy()

    for box, cls in zip(r.boxes.xyxy, r.boxes.cls):
        x1, y1, x2, y2 = map(int, box)
        class_name = model.names[int(cls)]

        # ✅ Thicker bounding box
        cv2.rectangle(
            img,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            3   # ⬅ increased thickness (was 1)
        )

        # ✅ Bigger & thicker text
        cv2.putText(
            img,
            class_name,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,     # ⬅ increased font size (was 0.4)
            (0, 255, 0),
            2,       # ⬅ increased thickness (was 1)
            cv2.LINE_AA
        )

 
   
    out_path = f"{save_dir}/result_{index}.jpg"
    cv2.imwrite(out_path, img)
    index += 1